# Economic Pulse — Colab Model Training

This notebook runs the **same version-controlled ML pipeline** used by the Economic Pulse repository. It does not contain a separate copy of the model logic.

**Flow:** sync GitHub source → install dependencies → upload CSV → validate vintages → train/backtest → inspect outputs → download evidence bundle → optionally push changed Python source files to GitHub.

The initial target is monthly UPI transaction counts, not GDP. Never invent historical `available_at` timestamps or revisions.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = "https://github.com/MAINAKRUDRA/ECONOMIC-PULSE.git"
REPO_DIR = Path("/content/ECONOMIC-PULSE")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Repository:", Path.cwd())
print("HEAD:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
assert (REPO_DIR / "packages/ml/train.py").is_file(), "ML training source is missing from the repository"

In [ ]:
%pip install -q -r requirements-training.txt
print("Training dependencies installed.")

## 1. Upload the training dataset

Required CSV columns:
`series_id, period, available_at, revision, volume_count`

`period` must be the first day of the reference month. `available_at` must be the real timestamp when that observation/vintage became available. If you only have a latest-value snapshot, do not fabricate a vintage history; the notebook will still validate the file, but historical as-of evaluation may be limited.

In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError(f"Upload exactly one CSV file; received {len(uploaded)} files")
uploaded_name = next(iter(uploaded))
DATA_PATH = Path("/content") / uploaded_name
shutil.move(uploaded_name, DATA_PATH)
print("Dataset:", DATA_PATH, "| bytes:", DATA_PATH.stat().st_size)

In [ ]:
import pandas as pd
from packages.ml.data import load_vintages

raw = load_vintages(DATA_PATH)
series_ids = sorted(raw["series_id"].unique().tolist())
print("Rows:", len(raw))
print("Series IDs:", series_ids)
print("Reference periods:", raw.period.min(), "through", raw.period.max())
print("Vintage rows per series:")
display(raw.groupby("series_id").agg(rows=("volume_count", "size"), periods=("period", "nunique"), first_available=("available_at", "min"), last_available=("available_at", "max")))

SERIES_ID = input("Enter the exact series_id to train: ").strip()
if SERIES_ID not in series_ids:
    raise ValueError(f"Unknown series_id {SERIES_ID!r}. Choose one of: {series_ids}")
SOURCE_NAME = input("Data source name (e.g. RBI Payment System Indicators): ").strip()
RIGHTS_TAG = input("Data rights/provenance tag (confirm you are allowed to use it): ").strip()
if not SOURCE_NAME or not RIGHTS_TAG:
    raise ValueError("Real-data training requires source name and rights/provenance tag")

selected = raw[raw.series_id == SERIES_ID]
month_counts = selected.groupby("period").size()
print("Selected series:", SERIES_ID)
print("Months:", selected.period.nunique(), "| Vintage rows:", len(selected))
print("Months with multiple vintages:", int((month_counts > 1).sum()))

## 2. Train and evaluate

The pipeline builds release-aware features, compares flat/trend/seasonal baselines with XGBoost, performs chronological validation, reserves a final holdout when sufficient history exists, and writes a model card and manifest. XGBoost is not automatically promoted unless it beats the selected baseline on the required validation sample.

In [ ]:
from datetime import datetime, timezone
import json
from packages.ml.train import run_training

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT_DIR = Path("/content/economic-pulse-runs") / f"{SERIES_ID}-{RUN_ID}"
result = run_training(DATA_PATH, OUT_DIR, SERIES_ID, SOURCE_NAME, RIGHTS_TAG)
print(json.dumps(result, indent=2, default=str))
print("Output directory:", OUT_DIR)

In [ ]:
forecast = json.loads((OUT_DIR / "forecast_rows.json").read_text())
backtest = json.loads((OUT_DIR / "backtest.json").read_text())
challenger = json.loads((OUT_DIR / "challenger_validation.json").read_text())
manifest = json.loads((OUT_DIR / "manifest.json").read_text())

print("FORECAST")
display(pd.DataFrame(forecast))
print("BACKTEST METRICS")
display(pd.DataFrame(backtest.get("metrics", {})).T)
print("CHALLENGER VALIDATION")
print(json.dumps(challenger, indent=2))
print("MODEL CARD")
print((OUT_DIR / "model_card.md").read_text())
print("MANIFEST SUMMARY")
print({k: manifest.get(k) for k in ["series_id", "source_name", "rights_tag", "training_rows", "training_cutoff", "selection", "candidate_status", "mode"]})

In [ ]:
from packages.ml.publish_bundle import validate_bundle
import shutil

validation = validate_bundle(OUT_DIR)
print("Bundle validation:", json.dumps(validation, indent=2))
ZIP_PATH = Path("/content") / f"economic-pulse-{SERIES_ID}-{RUN_ID}.zip"
shutil.make_archive(str(ZIP_PATH.with_suffix("")), "zip", root_dir=OUT_DIR)
print("Evidence ZIP:", ZIP_PATH)
files.download(str(ZIP_PATH))

## 3. Optional: push changed ML source code to GitHub

This step pushes **only** changes under `packages/ml/`, `tests/`, and `requirements-training.txt`. It does not upload your CSV, forecasts, model binaries, or other run artifacts. Add a Colab Secret named `GH_TOKEN` with a GitHub token that has **Contents: Read and write** access to this repository. The token is read at runtime and is not written into the notebook or Git config.

If you have not edited the Python source files in this Colab session, the step reports that there is nothing to push. For notebook-cell edits, use **File → Save a copy in GitHub** or commit the `.ipynb` separately; this source-sync cell does not sync the Drive notebook automatically.

In [ ]:
from google.colab import userdata
import base64

tracked_paths = ["packages/ml", "tests", "requirements-training.txt"]
status = subprocess.run(["git", "status", "--porcelain", "--", *tracked_paths], capture_output=True, text=True, check=True).stdout
if not status.strip():
    print("No ML source changes detected. Nothing to push.")
else:
    token = userdata.get("GH_TOKEN")
    if not token:
        raise RuntimeError("Add GH_TOKEN in Colab Secrets, then rerun this cell. No changes were pushed.")
    subprocess.run(["git", "add", "--", *tracked_paths], check=True)
    staged = subprocess.run(["git", "diff", "--cached", "--quiet"])
    if staged.returncode == 0:
        print("No staged source changes to commit.")
    elif staged.returncode == 1:
        subprocess.run(["git", "config", "user.name", "Economic Pulse Colab"], check=True)
        subprocess.run(["git", "config", "user.email", "colab-bot@users.noreply.github.com"], check=True)
        subprocess.run(["git", "commit", "-m", "Update Economic Pulse ML source from Colab"], check=True)
        auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        subprocess.run(["git", "-c", f"http.extraheader=AUTHORIZATION: basic {auth}", "push", "origin", "HEAD:main"], check=True)
        print("ML source changes pushed to GitHub main.")
    else:
        raise RuntimeError("Could not inspect staged Git changes")
    del token
    if "auth" in locals():
        del auth

## Important notes

- Colab runtime storage is temporary. Download the evidence ZIP before disconnecting.
- The GitHub Actions workflow tests the ML pipeline and trains a synthetic fixture when ML source files change; synthetic output is not real forecast evidence.
- The current model forecasts monthly transaction counts. GDP forecasting requires a separate target, dataset, features, and validation design.
- A successful training run is not proof of predictive accuracy. Review validation rows, holdout results, baseline comparisons, and data provenance before using forecasts.